# Module 03 — AI Agents
## Lesson 6 — Error Handling and Retries

**Core principle:** the host classifies failures and owns bounded retry policy. The model may decide what to do after a final failure becomes an observation, but it does not control infrastructure retries.


## Failure taxonomy

- **Transient infrastructure:** timeout, connection reset, 429, many 5xx responses — often retryable.
- **Permanent request / validation:** malformed arguments, invalid dates, unsupported tools — fail fast.
- **Domain failure:** no availability, not eligible, resource not found — the operation succeeded but the requested outcome is unavailable.
- **Permission / authentication:** 401/403/missing scope — fix authority rather than blindly retrying.

Retry only when repeating the same operation shortly has a realistic chance of succeeding.


## Retry at the narrowest owning layer

Provider-call retries belong around the provider call. HTTP/tool retries belong inside the tool adapter. Do not retry an entire agent step because one low-level request timed out.


In [ ]:
from dataclasses import dataclass
import time
from typing import Callable, TypeVar

T = TypeVar("T")

class TransientToolError(RuntimeError):
    pass

class PermanentToolError(RuntimeError):
    pass

@dataclass(frozen=True)
class RetryPolicy:
    max_attempts: int = 3
    base_delay_seconds: float = 0.25
    max_delay_seconds: float = 2.0

def retry_call(
    operation: Callable[[], T],
    *,
    policy: RetryPolicy,
    is_retryable: Callable[[Exception], bool],
    sleep: Callable[[float], None] = time.sleep,
) -> T:
    for attempt in range(1, policy.max_attempts + 1):
        try:
            return operation()
        except Exception as exc:
            if not is_retryable(exc) or attempt == policy.max_attempts:
                raise
            delay = min(
                policy.base_delay_seconds * (2 ** (attempt - 1)),
                policy.max_delay_seconds,
            )
            print(f"attempt {attempt} failed: {exc}; retrying in {delay:.2f}s")
            sleep(delay)
    raise AssertionError("unreachable")


## Example 1 — deterministic transient failure

The operation fails twice and then succeeds. We disable sleeping so the notebook is fast and reproducible.


In [ ]:
attempts = 0

def flaky_weather_lookup() -> dict:
    global attempts
    attempts += 1
    if attempts < 3:
        raise TransientToolError("weather API timed out")
    return {"city": "Melbourne", "rain_probability_percent": 35}

result = retry_call(
    flaky_weather_lookup,
    policy=RetryPolicy(max_attempts=3, base_delay_seconds=0),
    is_retryable=lambda exc: isinstance(exc, TransientToolError),
    sleep=lambda _: None,
)

print(result)
print("attempts:", attempts)


## Permanent failures fail fast

Retrying invalid input repeats the same error. The predicate excludes permanent validation failures.


In [ ]:
attempts = 0

def invalid_request() -> dict:
    global attempts
    attempts += 1
    raise PermanentToolError("target_date must be YYYY-MM-DD")

try:
    retry_call(
        invalid_request,
        policy=RetryPolicy(max_attempts=5, base_delay_seconds=0),
        is_retryable=lambda exc: isinstance(exc, TransientToolError),
        sleep=lambda _: None,
    )
except PermanentToolError as exc:
    print("failed immediately:", exc)
    print("attempts:", attempts)


## Example 2 — turn final failure into an agent observation

Low-level retries happen inside host code. Only the final result — success or exhausted failure — is added to agent state.


In [ ]:
def run_tool_with_policy(name: str, operation: Callable[[], dict]) -> dict:
    try:
        data = retry_call(
            operation,
            policy=RetryPolicy(max_attempts=3, base_delay_seconds=0),
            is_retryable=lambda exc: isinstance(exc, TransientToolError),
            sleep=lambda _: None,
        )
        return {"ok": True, "tool": name, "data": data}
    except PermanentToolError as exc:
        return {"ok": False, "tool": name, "error": {"kind": "validation", "retryable": False, "message": str(exc)}}
    except TransientToolError as exc:
        return {"ok": False, "tool": name, "error": {"kind": "transient_exhausted", "retryable": False, "message": str(exc)}}


## Provider calls have a separate retry boundary

When you use the OpenAI SDK, connection errors, timeouts, rate limits, and some server errors may be retryable. Keep this policy separate from tool retries and be aware of any retries already performed by the SDK.


In [ ]:
from openai import APIConnectionError, APITimeoutError, InternalServerError, RateLimitError

RETRYABLE_PROVIDER_ERRORS = (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    RateLimitError,
)

def call_model_with_retry(client, **kwargs):
    return retry_call(
        lambda: client.responses.create(**kwargs),
        policy=RetryPolicy(max_attempts=3, base_delay_seconds=0.5),
        is_retryable=lambda exc: isinstance(exc, RETRYABLE_PROVIDER_ERRORS),
    )


## Side effects and idempotency

A timeout does not prove that the server did nothing. If a booking, payment, email, deployment, or deletion may already have happened, a blind retry can duplicate the side effect. Use an idempotency key or query the operation status before retrying.

```python
create_booking(trip_id="trip-123", idempotency_key="agent-run-42:step-3")
```


## Retry budget is not the same as agent step budget

One agent step can contain several low-level HTTP attempts. Keep attempts, timeouts, model steps, tool calls, and cost visible as separate budgets.


## Exercises

1. Make the flaky tool succeed only on attempt 4 while the policy allows 3 attempts.
2. Implement an HTTP status classifier: retry 429 and 5xx; fail fast on 400, 401, 403, and 404.
3. Add jitter while injecting randomness so tests stay deterministic.
4. Design an idempotency key for a hypothetical `send_email` or `create_booking` tool.
5. Trace a full agent request and count model steps separately from low-level retry attempts.

Next: **Lesson 7 — Stopping Conditions and Budgets**.
